# UIT-ViQuAD2.0 — Qwen2.5-3B DeLoRA

Notebook chạy một pipeline hoàn chỉnh:

1. Fine-tune **DeLoRA** → `qwen2.5-3b-instruct-delora-viquad2`
2. Đánh giá EM/F1/NoAns trên toàn bộ `validation`
3. Suy luận toàn bộ test và ghi `results.json`

DeLoRA là `peft.DeloraConfig` (`peft_type="DELORA"`), không phải DoRA (`use_dora=True`). DeLoRA phải nạp base model BF16/FP16, không dùng BnB 4/8-bit vì bước khởi tạo tính norm trên trọng số gốc.

### Cấu hình RTX 3090 23 GB
- Micro-batch `1`, gradient accumulation `8` → effective batch `8`
- BF16 nếu hỗ trợ, nếu không dùng FP16; gradient checkpointing bật
- Tối đa 5 epochs; validation mỗi 200 optimizer steps
- Early stopping khi `eval_loss` không giảm ít nhất `0.001` trong 3 lần validation liên tiếp
- Tự khôi phục checkpoint có `eval_loss` thấp nhất

### Cách chạy
1. Chạy pip cells, sau đó **Restart Kernel**.
2. Chạy tuần tự tất cả cells từ đầu đến cuối.
3. Sau training, notebook tự tạo:
   `qwen2.5-3b-instruct-delora-viquad2/results.json`
4. Submission có schema nghiêm ngặt:
   `{"uit_id": "answer_of_model", ...}`; no-answer được ghi là chuỗi rỗng.

In [ ]:
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

In [ ]:
# Cài stack trước, sau đó pin PEFT cuối cùng để Unsloth không thay đổi version DeLoRA đã kiểm chứng.
# Không cần xformers: PyTorch SDPA ổn định hơn với wheel CUDA 12.8.
!pip install transformers trl accelerate bitsandbytes datasets --no-cache-dir
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --no-cache-dir
!pip install unsloth_zoo scikit-learn safetensors --no-cache-dir
!pip install "peft==0.19.1" --no-deps --force-reinstall --no-cache-dir


In [ ]:
import importlib
import inspect

for pkg in ["torch", "transformers", "datasets", "unsloth", "peft"]:
    importlib.import_module(pkg)
    print(f"OK  {pkg}")

import peft
from peft import DeloraConfig

print(f"PEFT version: {peft.__version__}")
if peft.__version__ != "0.19.1":
    raise RuntimeError(f"Cần peft==0.19.1, hiện tại là {peft.__version__}. Restart Kernel sau pip cell.")
sig = inspect.signature(DeloraConfig.__init__).parameters
required = {"r", "delora_lambda", "target_modules", "module_dropout"}
missing = required.difference(sig)
if missing:
    raise RuntimeError(f"DeloraConfig thiếu tham số: {sorted(missing)}")
print("DeLoRA API OK:", ", ".join(k for k in sig if k != "self"))

In [ ]:
import warnings, logging, os, sys
import torch

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["ACCELERATE_BYPASS_DEVICE_MAP"] = "true"
os.environ["ACCELERATE_NUM_PROCESSES"] = "1"
for _n in ("transformers", "datasets", "torch", "unsloth", "peft", "accelerate", "huggingface_hub"):
    logging.getLogger(_n).setLevel(logging.ERROR)
try:
    from transformers.utils import logging as hf_logging
    hf_logging.set_verbosity_error()
except Exception:
    pass

# RTX 3090 (Ampere) — bật TF32 + cuDNN autotune để tăng throughput matmul/conv
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

print("Warnings suppressed.")
if torch.cuda.is_available():
    print(f"CUDA speed: TF32={torch.backends.cuda.matmul.allow_tf32}, cudnn.benchmark={torch.backends.cudnn.benchmark}")

In [ ]:
import gc
import json
import math
import re
import string
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoTokenizer

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    cc = torch.cuda.get_device_capability(0)
    arch = "Ampere (3090)" if cc == (8, 6) else "Ada (4090)" if cc == (8, 9) else f"sm_{cc[0]}{cc[1]}"
    print(f"GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | {arch}")
    if cc == (8, 6):
        print("-> RTX 3090: DeLoRA 3B bf16 + TF32 + batch=1/grad_accum=8")
    elif cc == (8, 9):
        print("-> RTX 4090: DeLoRA 3B bf16 + TF32 + batch=1/grad_accum=8")

# ========== CẤU HÌNH 3B DELORA ==========
BASE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DATASET_NAME = "taidng/UIT-ViQuAD2.0"
NO_ANSWER_SENTINEL = "Không có đáp án trong đoạn văn"

# Prompt siết chặt → ép model trả span-only (tăng EM)
SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện trong đoạn văn, không thêm bất kỳ từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm 'Đáp án:', 'là', 'thủ đô'.\n"
    f"3) Nếu không có đáp án trong đoạn văn, chỉ trả về: {NO_ANSWER_SENTINEL}"
)

USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn dưới đây. "
    "Chỉ trả về cụm từ trong đoạn văn, không thêm từ nào khác.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    f"Câu trả lời (span-only hoặc '{NO_ANSWER_SENTINEL}'):"
)

PROFILING_CONFIG_PATH = "profiling_config_3b.json"

DATASET_ROOT = Path("../../")
DEV_JSON_PATH = DATASET_ROOT / "dev_viquad2.json"
TEST_JSON_PATH = DATASET_ROOT / "test_viquad2.json"

RUN_TRAINING = True
RUN_METRIC_EVAL = True
RUN_SUBMISSION_EXPORT = True
FORCE_HF_DATASET = True
FORCE_REEXPORT_JSON = True

COMPARE_EVAL_PATH = "eval_compare_delora_viquad2_3b.json"
EXPECTED_TEST_SIZE = 7301
EVAL_MAX_SAMPLES = None
SUBMISSION_MAX_SAMPLES = None
EVAL_LOG_EVERY = 10
SUBMISSION_LOG_EVERY = 50

# DeLoRA init tính norm(base.weight): bắt buộc base BF16/FP16, không dùng BnB 4/8-bit.
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
FORCE_FP16 = False
ATTN_IMPLEMENTATION = "auto"
MAX_SEQ_CAP = 4096
MIN_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 32
USE_SPAN_POSTPROCESS = True

# DeLoRA 3B full precision trên RTX 3090 23 GB: preset an toàn effective batch=8.
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
# Giữ để các nhánh legacy trong helper không NameError; pipeline hiện chỉ chọn DeLoRA.
LORA_R, LORA_ALPHA = 16, 32

# Early stopping: patience đếm LẦN VALIDATION, không phải epoch.
USE_EARLY_STOPPING = True
MAX_TRAIN_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001
EVAL_STEPS = 200

TRAIN_COMMON = dict(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_ratio=0.03,
    num_train_epochs=MAX_TRAIN_EPOCHS,
    learning_rate=1e-4,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
)

DATASET_NUM_PROC = 4
DATALOADER_NUM_WORKERS = 0
SAVE_STEPS = 200
SAVE_TOTAL_LIMIT = 4
RESUME_FROM_CHECKPOINT = False

ADAPTER_VARIANTS = [
    {"name": "delora", "save_path": "qwen2.5-3b-instruct-delora-viquad2", "output_dir": "outputs_viquad2_3b_delora"},
]
TRAIN_METHODS = ["delora"]
EVAL_METHODS = ["delora"]

TQDM_BAR = "{desc}: {percentage:3.0f}%|{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"

def resolve_attn_implementation(preferred=ATTN_IMPLEMENTATION):
    """flash_attention_2 nếu đã cài; không thì sdpa (nhanh trên Ampere/Ada)."""
    if preferred and preferred != "auto":
        return preferred
    try:
        import flash_attn  # noqa: F401
        return "flash_attention_2"
    except Exception:
        return "sdpa"


def filter_training_args(kwargs):
    """Tương thích transformers mới/cũ: eval_strategy vs evaluation_strategy."""
    import inspect
    from transformers import TrainingArguments

    sig = inspect.signature(TrainingArguments.__init__).parameters
    out = dict(kwargs)
    # Rename legacy -> new
    if "evaluation_strategy" in out and "eval_strategy" in sig and "evaluation_strategy" not in sig:
        out["eval_strategy"] = out.pop("evaluation_strategy")
    elif "eval_strategy" in out and "evaluation_strategy" in sig and "eval_strategy" not in sig:
        out["evaluation_strategy"] = out.pop("eval_strategy")
    # Drop unknown keys (avoid TypeError on version skew)
    return {k: v for k, v in out.items() if k == "self" or k in sig}



def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def get_variant(method_name):
    for variant in ADAPTER_VARIANTS:
        if variant["name"] == method_name:
            return variant
    raise ValueError(f"Unknown adapter method: {method_name}")


def build_messages(context, question, answer=None, for_inference=False):
    user_content = USER_PROMPT_TEMPLATE.format(context=context, question=question)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
    ]
    if answer is not None and not for_inference:
        messages.append({"role": "assistant", "content": answer})
    return messages


def sample_to_train_text(sample, tok):
    messages = build_messages(sample["context"], sample["question"], sample["answer"])
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


def load_tokenizer(model_path=BASE_MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(model_path)
    if tok.chat_template is None:
        raise RuntimeError(f"Tokenizer {model_path} thiếu chat_template.")
    # Qwen tokenizer đôi khi pad_token=None → dễ lỗi ở generate / trainer
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok

In [ ]:
def row_to_sample(row, *, allow_unlabeled=False):
    """Map HF row; hidden test labels may be None or empty answer lists."""
    context = str(row.get("context") or "").strip()
    question = str(row.get("question") or "").strip()
    if not context or not question:
        raise ValueError(f"Sample {row.get('uit_id') or row.get('id')} thiếu context/question")

    is_impossible = bool(row["is_impossible"])
    answers = row.get("answers")
    texts = (answers or {}).get("text") or []
    base = {
        "id": row.get("id", ""), "uit_id": row.get("uit_id", ""),
        "title": row.get("title", ""), "context": context, "question": question,
    }
    if allow_unlabeled and not texts:
        return {**base, "answer": None, "is_impossible": is_impossible, "has_label": False}
    if answers is None:
        return {**base, "answer": None, "is_impossible": is_impossible, "has_label": False}
    if is_impossible or not texts or not str(texts[0]).strip():
        return {**base, "answer": NO_ANSWER_SENTINEL, "is_impossible": True, "has_label": True}
    return {**base, "answer": str(texts[0]).strip(), "is_impossible": False, "has_label": True}


def hf_split_to_samples(split_dataset, *, allow_unlabeled=False):
    return [row_to_sample(row, allow_unlabeled=allow_unlabeled) for row in split_dataset]


def save_samples_json(samples, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    print(f"Saved {len(samples)} → {path.resolve()}")


def split_stats(name, samples):
    n = len(samples)
    n_no = sum(1 for s in samples if s["is_impossible"])
    print(f"{name}: {n} | NoAns: {n_no} ({100*n_no/max(n,1):.1f}%)")


try:
    raw_ds = load_dataset(DATASET_NAME)
    train_samples = hf_split_to_samples(raw_ds["train"])
    dev_samples = hf_split_to_samples(raw_ds["validation"])
    test_samples = hf_split_to_samples(raw_ds["test"], allow_unlabeled=True)
except Exception as e:
    if FORCE_HF_DATASET:
        raise RuntimeError(f"Không tải được HF: {e}")
    raise RuntimeError("Không fallback validation thành train vì sẽ gây data leakage") from e

if len(test_samples) != EXPECTED_TEST_SIZE:
    raise RuntimeError(f"Test split phải có {EXPECTED_TEST_SIZE} câu, nhận {len(test_samples)}")
if any(s.get("has_label", True) for s in test_samples):
    raise RuntimeError("Test split ẩn nhãn nhưng có sample bị đánh dấu has_label=True")
test_keys = [s.get("uit_id") for s in test_samples]
if any(not isinstance(k, str) or not k for k in test_keys) or len(set(test_keys)) != len(test_keys):
    raise RuntimeError("Test split có uit_id rỗng, sai kiểu hoặc trùng")

eval_samples = [s for s in dev_samples if s.get("has_label", True)]
split_stats("Train", train_samples)
split_stats("Val", dev_samples)
split_stats("Test", test_samples)
if FORCE_REEXPORT_JSON:
    save_samples_json(dev_samples, DEV_JSON_PATH)
    save_samples_json(test_samples, TEST_JSON_PATH)

In [ ]:
if "train_samples" not in globals():
    raise NameError("Chạy cell tải dataset trước.")


def compute_max_seq_length(samples, tok, cap=MAX_SEQ_CAP, min_len=MIN_SEQ_LENGTH):
    lengths = []
    total = len(samples)
    pbar = tqdm(samples, total=total, desc="Token profiling", unit="sample", bar_format=TQDM_BAR)
    for i, s in enumerate(pbar, 1):
        lengths.append(len(tok.encode(sample_to_train_text(s, tok))))
        pbar.set_postfix(done=f"{i}/{total}")
    lengths.sort()
    n = len(lengths)
    stats = {"min": lengths[0], "p50": lengths[n//2], "p95": lengths[int(n*0.95)],
             "p99": lengths[int(n*0.99)], "max": lengths[-1]}
    chosen = max(((min(math.ceil(stats["p99"]*1.05), cap)+255)//256)*256, min_len)
    stats.update({"chosen_max_seq_length": chosen,
                  "truncated_samples": sum(1 for L in lengths if L > chosen),
                  "truncated_pct": round(100*sum(1 for L in lengths if L > chosen)/n, 3)})
    return chosen, stats


tokenizer_prof = load_tokenizer()
if Path(PROFILING_CONFIG_PATH).exists() and not RUN_TRAINING:
    prof_cfg = json.load(open(PROFILING_CONFIG_PATH, encoding="utf-8"))
    max_seq_length, length_stats = prof_cfg["max_seq_length"], prof_cfg["token_length_stats"]
else:
    max_seq_length, length_stats = compute_max_seq_length(train_samples, tokenizer_prof)
    json.dump({"max_seq_length": max_seq_length, "token_length_stats": length_stats},
              open(PROFILING_CONFIG_PATH, "w", encoding="utf-8"), indent=2)
print(f"max_seq_length = {max_seq_length}")
for k, v in length_stats.items(): print(f"  {k}: {v}")

In [ ]:
# Format dataset 1 lần — dùng chung cho mọi adapter type
from datasets import Dataset

tokenizer_fmt = load_tokenizer()

def formatting_prompts_func(examples):
    texts = []
    for ctx, q, ans in zip(examples["context"], examples["question"], examples["answer"]):
        msgs = build_messages(ctx, q, ans)
        texts.append(tokenizer_fmt.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

train_hf = Dataset.from_list(train_samples)
try:
    dataset = train_hf.map(
        formatting_prompts_func,
        batched=True,
        batch_size=256,
        num_proc=DATASET_NUM_PROC,
        remove_columns=train_hf.column_names,
    )
except Exception as e:
    print(f"num_proc={DATASET_NUM_PROC} failed ({e}) → fallback num_proc=1")
    dataset = train_hf.map(
        formatting_prompts_func,
        batched=True,
        remove_columns=train_hf.column_names,
    )
print(f"Shared train dataset: {len(dataset)} samples")

eval_dataset = None
if USE_EARLY_STOPPING:
    dev_hf = Dataset.from_list(dev_samples)
    try:
        eval_dataset = dev_hf.map(
            formatting_prompts_func,
            batched=True,
            batch_size=256,
            num_proc=DATASET_NUM_PROC,
            remove_columns=dev_hf.column_names,
        )
    except Exception as e:
        print(f"Eval map num_proc failed ({e}) → fallback num_proc=1")
        eval_dataset = dev_hf.map(
            formatting_prompts_func,
            batched=True,
            remove_columns=dev_hf.column_names,
        )
    print(f"Eval dataset (early stopping): {len(eval_dataset)} samples")

## Train DeLoRA 3B

Pipeline: load base BF16/FP16 → gắn `DeloraConfig` → train với early stopping → khôi phục best `eval_loss` checkpoint → lưu adapter và tokenizer.

- `TRAIN_METHODS=["delora"]`: không train lại LoRA/TinyLoRA/DoRA hiện có.
- RTX 3090 23 GB dùng batch `1`, accumulation `8` để giữ effective batch `8`.
- Checkpoint/eval mỗi 200 steps; patience 3 nghĩa là dừng sau 3 lần validation liên tiếp không cải thiện đủ `0.001`.
- Có thể đặt `RESUME_FROM_CHECKPOINT=True` để tiếp tục checkpoint DeLoRA mới nhất.


In [ ]:
def resolve_resume_checkpoint(output_dir, resume_flag):
    """Trả về path checkpoint để resume, hoặc None nếu train từ đầu."""
    output_dir = Path(output_dir)
    if not resume_flag:
        return None
    if isinstance(resume_flag, str):
        ckpt = Path(resume_flag)
        if not ckpt.exists():
            raise FileNotFoundError(f"Checkpoint không tồn tại: {ckpt}")
        return str(ckpt)
    ckpts = sorted(output_dir.glob("checkpoint-*"), key=lambda p: int(p.name.rsplit("-", 1)[-1]))
    return str(ckpts[-1]) if ckpts else None


def apply_adapter(model, method_name):
    """Gắn PEFT DeLoRA lên CausalLM wrapper của base model Unsloth."""
    from unsloth import FastLanguageModel

    def _resolve_causallm(m):
        if hasattr(m, "prepare_inputs_for_generation"):
            return m
        if hasattr(m, "get_base_model"):
            base = m.get_base_model()
            if hasattr(base, "prepare_inputs_for_generation"):
                return base
        raise RuntimeError(
            "Không tìm thấy ForCausalLM. Đừng dùng model.model — đó là Qwen2Model "
            "(thiếu prepare_inputs_for_generation)."
        )

    if method_name == "lora":
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=TARGET_MODULES,
            lora_dropout=0.05, bias="none",
            use_gradient_checkpointing="unsloth", random_state=3407,
            use_dora=False,
        )
        print(f"Applied: LoRA (r={LORA_R}, alpha={LORA_ALPHA})")
        return model

    if method_name == "dora":
        model = FastLanguageModel.get_peft_model(
            model,
            r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=TARGET_MODULES,
            lora_dropout=0.05, bias="none",
            use_gradient_checkpointing="unsloth", random_state=3407,
            use_dora=True,
        )
        print(f"Applied: DoRA (r={LORA_R}, alpha={LORA_ALPHA}, use_dora=True)")
        return model

    if method_name == "tinylora":
        import inspect
        import peft
        from peft import TinyLoraConfig, get_peft_model as peft_get_model

        sig = inspect.signature(TinyLoraConfig.__init__).parameters
        desired = {
            "r": 2,
            "u": 64,
            "num_projections": 64,
            "target_modules": TARGET_MODULES,
            "tinylora_dropout": 0.0,
            "lora_dropout": 0.0,
            "bias": "none",
            "task_type": "CAUSAL_LM",
            "weight_tying": 0.0,
            "projection_seed": 3407,
            "init_weights": True,
        }
        tiny_kwargs = {k: v for k, v in desired.items() if k in sig}
        if "target_modules" not in tiny_kwargs:
            tiny_kwargs["target_modules"] = TARGET_MODULES

        try:
            tinylora_config = TinyLoraConfig(**tiny_kwargs)
        except Exception as e:
            raise RuntimeError(
                f"TinyLoRA không khả dụng với PEFT {peft.__version__}: {e}. "
                "Chạy lại pip cell (peft>=0.19) rồi Restart Kernel."
            ) from e

        causallm = _resolve_causallm(model)
        model = peft_get_model(causallm, tinylora_config)
        model.config.use_cache = False
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        model.print_trainable_parameters()
        u_val = tiny_kwargs.get("u", tiny_kwargs.get("num_projections", "?"))
        print(f"Applied: TinyLoRA (PEFT {peft.__version__}, r={tiny_kwargs.get('r')}, u={u_val})")
        return model

    if method_name == "delora":
        import peft
        from peft import DeloraConfig, get_peft_model as peft_get_model

        delora_config = DeloraConfig(
            r=16,
            delora_lambda=15,
            target_modules=TARGET_MODULES,
            module_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            init_weights=True,
        )
        model = peft_get_model(_resolve_causallm(model), delora_config)
        model.config.use_cache = False
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        model.print_trainable_parameters()
        print(f"Applied: DeLoRA (PEFT {peft.__version__}, r=16, delora_lambda=15)")
        return model

    raise ValueError(f"Unknown method: {method_name}")


def _resolve_load_and_train_dtype():
    """Chọn dtype load + fp16/bf16 train — phải khớp nhau (Unsloth strict)."""
    import torch
    from unsloth import is_bfloat16_supported

    if LOAD_IN_4BIT or LOAD_IN_8BIT:
        load_dtype = None
        use_bf16 = is_bfloat16_supported() and not FORCE_FP16
        use_fp16 = not use_bf16
    elif FORCE_FP16 or not is_bfloat16_supported():
        load_dtype = torch.float16
        use_fp16, use_bf16 = True, False
    else:
        load_dtype = None
        use_fp16, use_bf16 = False, True
    return load_dtype, use_fp16, use_bf16


def _sync_train_dtype_with_model(model, use_fp16, use_bf16):
    """Đọc dtype thực tế của model sau load/adapter — tránh TypeError Unsloth."""
    import torch

    param_dtype = next((p.dtype for p in model.parameters() if p.requires_grad or p.numel() > 0), None)
    if param_dtype == torch.float16:
        return True, False
    if param_dtype == torch.bfloat16:
        return False, True
    return use_fp16, use_bf16


def train_one_variant(variant, max_seq_length, dataset, eval_dataset=None, resume_flag=RESUME_FROM_CHECKPOINT):
    import inspect
    import torch
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from trl import SFTConfig, SFTTrainer
    from transformers import EarlyStoppingCallback
    import sys

    name = variant["name"]
    print("\n" + "=" * 60)
    print(f" TRAIN VARIANT: {name.upper()}")
    print("=" * 60)

    eval_on = USE_EARLY_STOPPING and eval_dataset is not None
    if USE_EARLY_STOPPING and eval_dataset is None:
        raise RuntimeError("USE_EARLY_STOPPING=True nhưng thiếu eval_dataset")
    if not torch.cuda.is_available():
        raise RuntimeError("Training Qwen2.5-3B chỉ được chạy trên CUDA GPU")

    clear_gpu()
    free_gib, total_gib = (x / 1024**3 for x in torch.cuda.mem_get_info())
    print(f"GPU preflight: free={free_gib:.2f}/{total_gib:.2f} GiB")
    if free_gib < 20.0:
        raise RuntimeError("DeLoRA 3B cần ít nhất 20 GiB VRAM trống. Restart kernel/đóng process GPU khác.")

    load_dtype, use_fp16, use_bf16 = _resolve_load_and_train_dtype()
    print(f"Dtype plan: load={load_dtype or 'auto'} | train={'fp16' if use_fp16 else 'bf16'}")
    print(f"Loading base model: {BASE_MODEL_NAME} (lần đầu có thể tải ~6GB, 5–15 phút)...", flush=True)

    attn_impl = resolve_attn_implementation()
    print(f"Attention: {attn_impl}", flush=True)
    load_kwargs = dict(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=load_dtype,
        load_in_4bit=False,
        load_in_8bit=False,
    )
    load_params = inspect.signature(FastLanguageModel.from_pretrained).parameters
    if "device_map" in load_params:
        load_kwargs["device_map"] = None
    if "attn_implementation" in load_params:
        load_kwargs["attn_implementation"] = attn_impl
    try:
        model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
    except TypeError as e:
        if "attn_implementation" not in str(e):
            raise
        print("[WARN] Unsloth không nhận attn_implementation; retry với mặc định", flush=True)
        load_kwargs.pop("attn_implementation", None)
        model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
    except Exception as e:
        if load_kwargs.get("attn_implementation") == "flash_attention_2":
            print(f"[WARN] FlashAttention failed ({e}); fallback sdpa", flush=True)
            load_kwargs["attn_implementation"] = "sdpa"
            model, tokenizer = FastLanguageModel.from_pretrained(**load_kwargs)
        else:
            raise
    model = model.to("cuda:0")
    print("Base model loaded.", flush=True)
    model = apply_adapter(model, name)

    use_fp16, use_bf16 = _sync_train_dtype_with_model(model, use_fp16, use_bf16)
    print(f"Dtype verified: train={'fp16' if use_fp16 else 'bf16'}")

    train_args = dict(
        **TRAIN_COMMON,
        fp16=use_fp16,
        bf16=use_bf16,
        output_dir=variant["output_dir"],
        disable_tqdm=False,
        logging_strategy="steps",
        logging_steps=SAVE_STEPS,
        logging_first_step=False,
        log_level="error",
        log_level_replica="error",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        dataloader_pin_memory=True,
    )
    callbacks = []
    if eval_on:
        train_args.update(
            eval_strategy="steps",  # transformers mới; filter_training_args map legacy
            evaluation_strategy="steps",
            eval_steps=EVAL_STEPS,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        )
        callbacks.append(EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        ))
        print(
            f"Early stopping ON: max_epochs={MAX_TRAIN_EPOCHS}, patience={EARLY_STOPPING_PATIENCE}, "
            f"eval every {EVAL_STEPS} steps on {len(eval_dataset)} val samples"
        )
    else:
        train_args["eval_strategy"] = "no"
        train_args["evaluation_strategy"] = "no"

    # TRL mới đặt max_length/dataset options trong SFTConfig, không nằm ở SFTTrainer.__init__.
    config_params = inspect.signature(SFTConfig.__init__).parameters
    if "eval_strategy" in config_params:
        train_args.pop("evaluation_strategy", None)
    elif "evaluation_strategy" in config_params:
        train_args["evaluation_strategy"] = train_args.pop("eval_strategy", "steps" if eval_on else "no")
    if "max_length" in config_params:
        train_args["max_length"] = max_seq_length
    elif "max_seq_length" in config_params:
        train_args["max_seq_length"] = max_seq_length
    for key, value in (
        ("dataset_text_field", "text"),
        ("dataset_num_proc", DATASET_NUM_PROC),
        ("packing", False),
    ):
        if key in config_params:
            train_args[key] = value
    sft_config_kwargs = {k: v for k, v in train_args.items() if k in config_params}
    sft_config = SFTConfig(**sft_config_kwargs)
    configured_length = getattr(sft_config, "max_length", getattr(sft_config, "max_seq_length", None))
    if configured_length != max_seq_length:
        raise RuntimeError(f"SFTConfig sequence length sai: {configured_length} != {max_seq_length}")

    sft_kwargs = dict(
        model=model,
        train_dataset=dataset,
        eval_dataset=eval_dataset if eval_on else None,
        args=sft_config,
        callbacks=callbacks,
    )
    sft_params = inspect.signature(SFTTrainer.__init__).parameters
    if "processing_class" in sft_params:
        sft_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in sft_params:
        sft_kwargs["tokenizer"] = tokenizer
    trainer = SFTTrainer(**sft_kwargs)
    cfg_cls, tr_cls = type(trainer.args), type(trainer)
    sys.modules["trl.trainer.sft_config"] = sys.modules[cfg_cls.__module__]
    sys.modules["trl.trainer.sft_trainer"] = sys.modules[tr_cls.__module__]
    sys.modules[cfg_cls.__module__].SFTConfig = cfg_cls
    sys.modules[tr_cls.__module__].SFTTrainer = tr_cls

    effective_batch = trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps
    steps_per_epoch = math.ceil(len(dataset) / max(effective_batch, 1))
    total_steps = int(steps_per_epoch * trainer.args.num_train_epochs)
    print(
        f"Planned: batch={trainer.args.per_device_train_batch_size}×accum={trainer.args.gradient_accumulation_steps} "
        f"(effective={effective_batch}) | max_epochs={trainer.args.num_train_epochs} | "
        f"steps/epoch≈{steps_per_epoch} | total_steps≈{total_steps}"
        + (f" | early_stop patience={EARLY_STOPPING_PATIENCE}" if eval_on else "")
    )

    resume_ckpt = resolve_resume_checkpoint(variant["output_dir"], resume_flag)
    if resume_ckpt:
        print(f"Resume from checkpoint: {resume_ckpt}")
        train_result = trainer.train(resume_from_checkpoint=resume_ckpt)
    else:
        if resume_flag:
            print("RESUME_FROM_CHECKPOINT=True nhưng chưa có checkpoint — train từ đầu.")
        train_result = trainer.train()

    print("Training metrics:", train_result.metrics)
    print("Best checkpoint:", trainer.state.best_model_checkpoint)
    print("Best eval_loss:", trainer.state.best_metric)

    save_path = Path(variant["save_path"])
    save_path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    saved_cfg = json.load(open(save_path / "adapter_config.json", encoding="utf-8"))
    if (saved_cfg.get("peft_type") or "").upper() != "DELORA":
        raise RuntimeError(f"Adapter lưu sai peft_type: {saved_cfg.get('peft_type')}")
    if saved_cfg.get("r") != 16 or saved_cfg.get("delora_lambda") != 15:
        raise RuntimeError("Adapter DeLoRA lưu sai r hoặc delora_lambda")
    print(f"Saved verified DeLoRA adapter → {save_path}")

    del trainer, model, tokenizer
    clear_gpu()
    return str(save_path)


In [ ]:
if RUN_TRAINING:
    variant_map = {v["name"]: v for v in ADAPTER_VARIANTS}
    trained_paths = {}
    for method in TRAIN_METHODS:
        if method not in variant_map:
            raise ValueError(f"Unknown TRAIN_METHODS item: {method}")
        path = train_one_variant(
            variant_map[method], max_seq_length, dataset,
            eval_dataset=eval_dataset if USE_EARLY_STOPPING else None,
        )
        trained_paths[method] = path
    print("\n✅ Train xong:")
    for k, v in trained_paths.items():
        print(f"  {k}: {v}")
else:
    trained_paths = {}
    print("RUN_TRAINING=False — bỏ qua train.")

## DeLoRA evaluation and submission

Cell tiếp theo chạy inference trong process Transformers+PEFT riêng (không import Unsloth), đánh giá full validation, rồi xuất full test theo schema `{uit_id: answer}`.

In [ ]:
# One clean subprocess loads Qwen2.5-3B + DeLoRA once, then handles validation and test.
_INFER_SCRIPT_3B = r'''
from __future__ import annotations
import argparse, json, os, re, string, sys, time, unicodedata
from collections import Counter
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
NO_ANSWER_SENTINEL = "Không có đáp án trong đoạn văn"
SYSTEM_PROMPT = (
    "Bạn là hệ thống trích xuất câu trả lời từ văn bản tiếng Việt. "
    "QUY TẮC BẮT BUỘC:\n"
    "1) Chỉ trả về ĐÚNG cụm từ xuất hiện trong đoạn văn, không thêm bất kỳ từ nào khác.\n"
    "2) Không viết câu hoàn chỉnh, không giải thích, không thêm 'Đáp án:', 'là', 'thủ đô'.\n"
    f"3) Nếu không có đáp án trong đoạn văn, chỉ trả về: {NO_ANSWER_SENTINEL}"
)
USER_PROMPT_TEMPLATE = (
    "Trích xuất câu trả lời từ đoạn văn dưới đây. "
    "Chỉ trả về cụm từ trong đoạn văn, không thêm từ nào khác.\n\n"
    "Đoạn văn:\n{context}\n\n"
    "Câu hỏi: {question}\n\n"
    f"Câu trả lời (span-only hoặc '{NO_ANSWER_SENTINEL}'):"
)
PREFIX_RE = re.compile(r"^(đáp án|answer|câu trả lời|theo đoạn văn|trong đoạn văn)\s*[:\-]?\s*", re.I)

def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())

def compute_f1(pred, truth):
    pt, tt = normalize_text(pred).split(), normalize_text(truth).split()
    if not pt and not tt: return 1.0
    if not pt or not tt: return 0.0
    common = Counter(pt) & Counter(tt)
    n = sum(common.values())
    if not n: return 0.0
    p, r = n / len(pt), n / len(tt)
    return 2 * p * r / (p + r)

def is_no_answer(pred):
    return normalize_text(pred) == normalize_text(NO_ANSWER_SENTINEL)

def clean_prediction(raw):
    pred = raw.strip().split("\n")[0].strip().strip("\"'")
    return PREFIX_RE.sub("", pred).strip()

def align_to_context(pred, context):
    if not pred or is_no_answer(pred): return pred
    idx = context.lower().find(pred.lower())
    if idx >= 0: return context[idx:idx + len(pred)]
    words, pred_words = context.split(), normalize_text(pred).split()
    if not pred_words: return pred
    n = len(pred_words)
    best_span, best_f1 = pred, 0.0
    for i in range(len(words)):
        for j in range(i + 1, min(i + n + 4, len(words)) + 1):
            span = " ".join(words[i:j])
            score = compute_f1(span, pred)
            if score > best_f1: best_f1, best_span = score, span
    return best_span.strip() if best_f1 >= 0.5 else pred

def main():
    p = argparse.ArgumentParser()
    p.add_argument("--adapter-dir", required=True)
    p.add_argument("--samples-json", required=True)
    p.add_argument("--output", required=True)
    p.add_argument("--base-model", required=True)
    p.add_argument("--max-seq-length", type=int, required=True)
    p.add_argument("--max-new-tokens", type=int, default=32)
    p.add_argument("--log-every", type=int, default=50)
    args = p.parse_args()
    if "unsloth" in sys.modules:
        raise RuntimeError("Subprocess inference không được import unsloth")

    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel, DeloraConfig  # noqa: F401

    adapter_dir = Path(args.adapter_dir)
    cfg = json.load(open(adapter_dir / "adapter_config.json", encoding="utf-8"))
    peft_type = (cfg.get("peft_type") or "").upper()
    if peft_type != "DELORA":
        raise RuntimeError(f"Expected DELORA adapter, got {peft_type!r}")
    base_model = cfg.get("base_model_name_or_path") or args.base_model
    if str(base_model).lower().startswith("unsloth/"):
        print(f"[Sub] remap {base_model} -> {args.base_model}", flush=True)
        base_model = args.base_model

    samples = json.load(open(args.samples_json, encoding="utf-8"))
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device != "cuda": raise RuntimeError("Inference 3B requires CUDA")
    tokenizer_source = str(adapter_dir) if (adapter_dir / "tokenizer_config.json").exists() else base_model
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
    model = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=dtype).to(device)
    model = PeftModel.from_pretrained(model, str(adapter_dir), is_trainable=False).to(device)
    model.config.use_cache = True
    model.eval()
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    preds, t0, total = [], time.time(), len(samples)
    for i, sample in enumerate(samples, 1):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT_TEMPLATE.format(
                context=sample["context"], question=sample["question"]
            )},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=args.max_seq_length).to(device)
        with torch.no_grad():
            generated = model.generate(
                input_ids=inputs["input_ids"], attention_mask=inputs.get("attention_mask"),
                max_new_tokens=args.max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
            )
        raw = tokenizer.decode(generated[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        pred_raw = clean_prediction(raw)
        pred = align_to_context(pred_raw, sample["context"])
        preds.append({
            "id": sample.get("id", ""), "uit_id": sample.get("uit_id", ""),
            "is_impossible": sample.get("is_impossible"), "has_label": sample.get("has_label", False),
            "ground_truth": sample.get("answer"), "prediction_raw": pred_raw, "prediction": pred,
        })
        if i == 1 or i % args.log_every == 0 or i == total:
            elapsed = time.time() - t0
            rate = i / max(elapsed, 1e-3)
            print(f"[Sub] {i}/{total} | {elapsed/60:.1f}m | ETA {(total-i)/max(rate,1e-3)/60:.1f}m", flush=True)
    with open(args.output, "w", encoding="utf-8") as f:
        json.dump(preds, f, ensure_ascii=False)
    print(f"[Sub] Saved {len(preds)} predictions", flush=True)

if __name__ == "__main__": main()
'''


def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())


def compute_em(pred, truth):
    return int(normalize_text(pred) == normalize_text(truth))


def compute_f1(pred, truth):
    pt, tt = normalize_text(pred).split(), normalize_text(truth).split()
    if not pt and not tt: return 1.0
    if not pt or not tt: return 0.0
    common = Counter(pt) & Counter(tt)
    n = sum(common.values())
    if not n: return 0.0
    precision, recall = n / len(pt), n / len(tt)
    return 2 * precision * recall / (precision + recall)


def is_no_answer(pred):
    return normalize_text(pred) == normalize_text(NO_ANSWER_SENTINEL)


def run_delora_inference(adapter_path, samples):
    import shutil, subprocess, tempfile
    script_path = Path("eval_infer_subprocess_3b.py")
    script_path.write_text(_INFER_SCRIPT_3B, encoding="utf-8")
    tmpdir = Path(tempfile.mkdtemp(prefix="viquad3b_delora_"))
    samples_path, preds_path = tmpdir / "samples.json", tmpdir / "preds.json"
    json.dump(samples, open(samples_path, "w", encoding="utf-8"), ensure_ascii=False)
    cmd = [
        sys.executable, str(script_path), "--adapter-dir", str(adapter_path),
        "--samples-json", str(samples_path), "--output", str(preds_path),
        "--base-model", BASE_MODEL_NAME, "--max-seq-length", str(max_seq_length),
        "--max-new-tokens", str(MAX_NEW_TOKENS), "--log-every", str(SUBMISSION_LOG_EVERY),
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in proc.stdout: print(line.rstrip(), flush=True)
    finally:
        proc.wait()
    if proc.returncode != 0:
        shutil.rmtree(tmpdir, ignore_errors=True)
        raise RuntimeError(f"DeLoRA inference subprocess failed: exit {proc.returncode}")
    predictions = json.load(open(preds_path, encoding="utf-8"))
    shutil.rmtree(tmpdir, ignore_errors=True)
    return predictions


adapter_path = Path("qwen2.5-3b-instruct-delora-viquad2")
if not (adapter_path / "adapter_model.safetensors").exists():
    raise FileNotFoundError(f"Thiếu DeLoRA adapter: {adapter_path}")

selected_eval = eval_samples if EVAL_MAX_SAMPLES is None else eval_samples[:EVAL_MAX_SAMPLES]
combined_samples = selected_eval + test_samples
clear_gpu()
all_predictions = run_delora_inference(adapter_path, combined_samples)
if len(all_predictions) != len(combined_samples):
    raise RuntimeError("Inference trả sai số lượng predictions")
eval_predictions = all_predictions[:len(selected_eval)]
test_predictions = all_predictions[len(selected_eval):]

if RUN_METRIC_EVAL:
    has_em, has_f1, no_ok = [], [], []
    for row in eval_predictions:
        pred, truth = row["prediction"], row["ground_truth"]
        if row["is_impossible"]:
            no_ok.append(int(is_no_answer(pred)))
        else:
            has_em.append(compute_em(pred, truth))
            has_f1.append(compute_f1(pred, truth))
    metrics = {
        "method": "delora", "adapter": str(adapter_path),
        "hasans_em": round(100 * sum(has_em) / max(len(has_em), 1), 4),
        "hasans_f1": round(100 * sum(has_f1) / max(len(has_f1), 1), 4),
        "noans_accuracy": round(100 * sum(no_ok) / max(len(no_ok), 1), 4),
        "n_hasans": len(has_em), "n_noans": len(no_ok), "total": len(eval_predictions),
    }
    json.dump({"metrics": metrics, "predictions": eval_predictions},
              open(COMPARE_EVAL_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    print("Validation metrics:", metrics)

if RUN_SUBMISSION_EXPORT:
    results = {}
    for row in test_predictions:
        key = row.get("uit_id")
        if not isinstance(key, str) or not key: raise RuntimeError("Prediction thiếu uit_id hợp lệ")
        if key in results: raise RuntimeError(f"Prediction trùng uit_id: {key}")
        pred = (row.get("prediction") or "").strip()
        results[key] = "" if is_no_answer(pred) else pred
    if len(results) != EXPECTED_TEST_SIZE:
        raise RuntimeError(f"results.json phải có {EXPECTED_TEST_SIZE} keys, nhận {len(results)}")
    if any(not isinstance(v, str) for v in results.values()):
        raise RuntimeError("results.json phải có schema {uit_id: answer_string}")
    results_path = adapter_path / "results.json"
    json.dump(results, open(results_path, "w", encoding="utf-8"), ensure_ascii=False, indent=4)
    if json.load(open(results_path, encoding="utf-8")) != results:
        raise RuntimeError("Read-back validation results.json thất bại")
    n_empty = sum(v == "" for v in results.values())
    print(f"Saved {results_path} | total={len(results)} | noAns={n_empty}")
    print('Schema OK: {"uit_id": "answer_of_model"}')